In [3]:
from ortools.sat.python import cp_model

In [9]:
from ortools.sat.python import cp_model

# Define job durations and worker limits
job_durations = {
    "J1": 4,
    "J2": 3,
    "J3": 2,
    "J4": 5,
}
worker_limits = {
    "W1": 6,
    "W2": 8,
    "W3": 5,
}
jobs = list(job_durations.keys())
workers = list(worker_limits.keys())

# Create the model
model = cp_model.CpModel()

# Decision variables: x[job][worker] = 1 if job is assigned to worker, else 0
x = {}
for job in jobs:
    for worker in workers:
        x[job, worker] = model.NewBoolVar(f"x_{job}_{worker}")

# Additional variable: workload[worker] represents the total time assigned to each worker
workload = {}
for worker in workers:
    workload[worker] = model.NewIntVar(0, max(worker_limits.values()), f"workload_{worker}")

# Objective variable: max_workload represents the maximum workload across all workers
max_workload = model.NewIntVar(0, max(worker_limits.values()), "max_workload")

# Constraints
# 1. Each job must be assigned to exactly one worker
for job in jobs:
    model.Add(sum(x[job, worker] for worker in workers) == 1)

# 2. Each worker can do at most two jobs
for worker in workers:
    model.Add(sum(x[job, worker] for job in jobs) <= 2)

# 3. Workers' total workload must not exceed their maximum available time
for worker in workers:
    model.Add(
        sum(job_durations[job] * x[job, worker] for job in jobs) <= worker_limits[worker]
    )

# 4. Calculate the total workload for each worker
for worker in workers:
    model.Add(
        sum(job_durations[job] * x[job, worker] for job in jobs) == workload[worker]
    )

# 5. Ensure that J1 and J2 are not assigned to the same worker
model.Add(sum(x["J1", worker] + x["J2", worker] for worker in workers) <= 1)

# 6. Define the objective: minimize the maximum workload
model.AddMaxEquality(max_workload, [workload[worker] for worker in workers])
model.Minimize(max_workload)

# Solve the model
solver = cp_model.CpSolver()
status = solver.Solve(model)

# Output results
if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print("Optimal Assignment:")
    for job in jobs:
        for worker in workers:
            if solver.Value(x[job, worker]) == 1:
                print(f"{job} -> {worker}")
    print("\nWorkloads:")
    for worker in workers:
        print(f"{worker}: {solver.Value(workload[worker])} hours")
    print(f"\nMaximum Workload: {solver.Value(max_workload)} hours")
else:
    print("No feasible solution found.")

No feasible solution found.


In [27]:
jobs = ['j1', 'j2', 'j3']
machines = [0, 1]  # M1=0, M2=1
time_slots = [0, 1, 2, 3, 4]  # T1=0, T2=1, T3=2

model = cp_model.CpModel()

job_machines = {}
job_times = {}

for job in jobs:
    job_machines[job] = model.NewIntVar(0, 1, f"{job}_machine")
    job_times[job] = model.NewIntVar(0, 2, f"{job}_time")

#No two jobs can run on the same machine at the same time.
for i in range(len(job)):
    for j in range(i+1, len(job)):
        j1 = jobs[i]
        j2 = jobs[j]
        #if machine same then time must differ
        # Create a BoolVar that is true if the machines are the same
        same_machine = model.NewBoolVar(f"{j1}_{j2}_same_machine")
        model.Add(job_machines[j1] == job_machines[j2]).OnlyEnforceIf(same_machine)
        model.Add(job_machines[j1] != job_machines[j2]).OnlyEnforceIf(same_machine.Not())

        # If jobs are on the same machine, then their time slots must differ
        model.Add(job_times[j1] != job_times[j2]).OnlyEnforceIf(same_machine)

# Constraint 2: J1 and J2 on different machines
model.add(job_machines['j1'] != job_machines['j2'])

# Constraint 3: J3 must be in time slot T2 or T3 => values 1 or 2
model.AddAllowedAssignments([job_times['j3']], [[1], [2]])

solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.FEASIBLE:
    for job in job_names:
        print(f"{job}: Machine M{solver.Value(job_machines[job]) + 1}, Time T{solver.Value(job_times[job]) + 1}")
else:
    print("No solution found.")

No solution found.


In [36]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()

jobs = ['J1', 'J2', 'J3']
job_machines = {}
job_times = {}

# Create variables
for job in jobs:
    job_machines[job] = model.NewIntVar(0, 1, f"{job}_machine")  # 2 machines
    job_times[job] = model.NewIntVar(0, 2, f"{job}_time")        # 3 time slots

# Constraint: If two jobs are on the same machine, they must have different time slots
for i in range(len(jobs)):
    for j in range(i + 1, len(jobs)):
        j1 = jobs[i]
        j2 = jobs[j]

        # BoolVar: same machine?
        same_machine = model.NewBoolVar(f"{j1}_{j2}_same_machine")
        model.Add(job_machines[j1] == job_machines[j2]).OnlyEnforceIf(same_machine)
        model.Add(job_machines[j1] != job_machines[j2]).OnlyEnforceIf(same_machine.Not())

        # If same machine => times must differ
        model.Add(job_times[j1] != job_times[j2]).OnlyEnforceIf(same_machine)

# Optional constraint: J1 and J2 must be on different machines
# Uncomment this line only if needed
#model.Add(job_machines['J1'] != job_machines['J2'])

# Solve
solver = cp_model.CpSolver()
status = solver.Solve(model)

# Output
if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    for job in jobs:
        print(f"{job}: Machine {solver.Value(job_machines[job])}, Time Slot {solver.Value(job_times[job])}")
else:
    print("No solution found.")


J1: Machine 0, Time Slot 0
J2: Machine 0, Time Slot 1
J3: Machine 1, Time Slot 0


In [2]:
# Define product and slot classes
class Product:
    def __init__(self, name, frequency, volume):
        self.name = name
        self.frequency = frequency
        self.volume = volume

class Slot:
    def __init__(self, name, distance, capacity=1):
        self.name = name
        self.distance = distance
        self.capacity = capacity
        self.filled = 0

# Lower score is better. High-frequency items should be in low-distance slots
def score_assignment(product, slot):
    return product.frequency * slot.distance

#Assign products to slots to minimize total walking distance while satisfying capacity
def assign_products_to_slots(products, slots):
    assignments = {}
    used_slots = set()

    # Sort products by frequency (high to low)
    sorted_products = sorted(products, key=lambda p: -p.frequency)
    # Sort slots by proximity (low to high distance)
    sorted_slots = sorted(slots, key=lambda s: s.distance)

    for product in sorted_products:
        for slot in sorted_slots:
            if slot.capacity - slot.filled >= product.volume:
                assignments[product.name] = slot.name
                slot.filled += product.volume
                used_slots.add(slot.name)
                break

    return assignments

def total_walking_distance(assignments, products, slots):
    slot_map = {s.name: s.distance for s in slots}
    product_map = {p.name: p for p in products}
    total = 0

    for product_name, slot_name in assignments.items():
        product = product_map[product_name]
        dist = slot_map[slot_name]
        total += product.frequency * dist

    return total

if __name__ == "__main__":
    
    products = [
        Product("Product 1", frequency=15, volume=2),
        Product("Product 2", frequency=8, volume=1),
        Product("Product 3", frequency=20, volume=3),
    ]

    slots = [
        Slot("Slot 1", distance=1, capacity=3),
        Slot("Slot 2", distance=2, capacity=2),
        Slot("Slot 3", distance=3, capacity=3),
    ]

    assignment = assign_products_to_slots(products, slots)

    print("Product-to-slot assignments:")
    for product, slot in assignment.items():
        print(f"{product} → {slot}")

    total_cost = total_walking_distance(assignment, products, slots)
    print(f"\nTotal walking distance score: {total_cost}")


Product-to-slot assignments:
Product 3 → Slot 1
Product 1 → Slot 2
Product 2 → Slot 3

Total walking distance score: 74


In [ ]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()

jobs = ['J1', 'J2', 'J3']
job_machines = {}
job_times = {}

# Create variables
for job in jobs:
    job_machines[job] = model.NewIntVar(0, 1, f"{job}_machine")  # 2 machines
    job_times[job] = model.NewIntVar(0, 2, f"{job}_time")        # 3 time slots

# Constraint: If two jobs are on the same machine, they must have different time slots
for i in range(len(jobs)):
    for j in range(i + 1, len(jobs)):
        j1 = jobs[i]
        j2 = jobs[j]

        # BoolVar: same machine?
        same_machine = model.NewBoolVar(f"{j1}_{j2}_same_machine")
        model.Add(job_machines[j1] == job_machines[j2]).OnlyEnforceIf(same_machine)
        model.Add(job_machines[j1] != job_machines[j2]).OnlyEnforceIf(same_machine.Not())

        # If same machine => times must differ
        model.Add(job_times[j1] != job_times[j2]).OnlyEnforceIf(same_machine)

# Optional constraint: J1 and J2 must be on different machines
# Uncomment this line only if needed
#model.Add(job_machines['J1'] != job_machines['J2'])

# Solve
solver = cp_model.CpSolver()
status = solver.Solve(model)

# Output
if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    for job in jobs:
        print(f"{job}: Machine {solver.Value(job_machines[job])}, Time Slot {solver.Value(job_times[job])}")
else:
    print("No solution found.")